# 00: Common Imports and Setup

This notebook contains common imports, setup, and configuration that is shared across all notebooks.

**Usage**: Run this notebook first using `%run 00-import.ipynb` at the start of other notebooks.


## Standard Library Imports


In [1]:
# Enable automatic module reloading for development
# This ensures code changes are picked up without restarting the kernel
%load_ext autoreload
%autoreload 2

# Standard library imports
import sys
import os
from pathlib import Path

# Add project root to path
# In Jupyter, we need to find the project root more reliably
# Try multiple methods to find the project root

# Method 1: Try to use __file__ if available (when running as script)
try:
    notebook_file = Path(__file__).resolve()
    if notebook_file.parent.name == "notebooks":
        project_root = notebook_file.parent.parent
    else:
        project_root = notebook_file.parent
except NameError:
    # __file__ not available in Jupyter, use current working directory
    current_dir = Path(os.getcwd())
    
    # Method 2: If we're in notebooks directory, go up one level
    if current_dir.name == "notebooks":
        project_root = current_dir.parent
    # Method 3: If we're in the project root, use it directly
    elif (current_dir / "src" / "knowledge_agents").exists():
        project_root = current_dir
    # Method 4: Walk up the directory tree to find project root
    else:
        project_root = current_dir
        while project_root != project_root.parent:
            if (project_root / "src" / "knowledge_agents").exists():
                break
            project_root = project_root.parent
        else:
            # Fallback: assume we're in notebooks directory
            project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

# Add src to path
src_path = project_root / "src"
if src_path.exists() and (src_path / "knowledge_agents").exists():
    if str(src_path) not in sys.path:
        sys.path.insert(0, str(src_path))
    print(f"✅ Added to path: {src_path}")
    print(f"   Project root: {project_root}")
else:
    print(f"⚠️  Warning: src/knowledge_agents directory not found at {src_path}")
    print(f"   Current directory: {os.getcwd()}")
    print(f"   Project root: {project_root}")
    print(f"   Please ensure you're running from the correct directory")

print("✅ Standard library imports loaded")


✅ Added to path: /Users/omareid/Workspace/git/knowledge-agents/src
   Project root: /Users/omareid/Workspace/git/knowledge-agents
✅ Standard library imports loaded


## Repository Components


In [2]:
# Import repository components
try:
    from knowledge_agents.config.api_config import Settings, get_settings
    from knowledge_agents.dependencies import Dependencies
    from knowledge_agents.clients.neo4j_client import Neo4jClientManager
    print("✅ Repository components imported")
except ImportError as e:
    print(f"❌ Error importing repository components: {e}")
    print(f"   Current working directory: {os.getcwd()}")
    print(f"   Python path: {sys.path[:3]}")  # Show first 3 entries
    print(f"   Please ensure you're running this from the notebooks directory")
    print(f"   or that the project structure is correct")
    raise


✅ Repository components imported


## Environment Detection and Settings


In [3]:
# Detect if running in container (for NotePlan directory selection)
IS_CONTAINER = Path("/noteplan").exists()

# Force reload the module to pick up any code changes
# This is more reliable than relying solely on autoreload
import importlib
try:
    import knowledge_agents.config.api_config
    importlib.reload(knowledge_agents.config.api_config)
except ImportError:
    pass  # Module not loaded yet, will be imported below

# Reset settings cache to ensure we get fresh instance with updated defaults
try:
    from knowledge_agents.config.api_config import reset_settings, is_running_in_container
    reset_settings()
except ImportError as e:
    # If import still fails, try one more reload and provide helpful message
    print(f"⚠️  Import failed: {e}")
    print("   Attempting explicit reload...")
    import knowledge_agents.config.api_config
    importlib.reload(knowledge_agents.config.api_config)
    from knowledge_agents.config.api_config import reset_settings, is_running_in_container
    reset_settings()
    print("✅ Reload successful!")

# Load settings with runtime-aware defaults
# See Settings class docstring for full precedence order and usage examples
# Quick reference:
# - Manual overrides (kwargs) > Environment variables (.env file) > Runtime-aware defaults > Field defaults
# - Use runtime_env='container' or 'local' to force runtime defaults
# - All settings can be overridden via kwargs or environment variables
#
# Note: If your .env file has incorrect values, you can override them here:
# settings = get_settings(neo4j_password='admin123')  # Override .env file value
settings = get_settings()

# Debug output
runtime_detected = "container" if is_running_in_container() else "local"
print(f"🔍 Runtime detection: {runtime_detected}")
print(f"✅ Settings loaded:")
print(f"   Neo4j URI: {settings.neo4j_uri}")
print(f"   Neo4j Database: {settings.neo4j_database}")
print(f"   Neo4j Username: {settings.neo4j_username}")
print(f"   Neo4j Password: {'*' * len(settings.neo4j_password) if settings.neo4j_password else 'not set'}")
print(f"   LiteLLM Proxy Host: {settings.litellm_proxy_host}")
print(f"\n💡 To override settings, see Settings class docstring:")
print(f"   help(Settings)  # or help(get_settings)")
print(f"   # Quick examples:")
print(f"   # settings = get_settings(neo4j_password='your_password')")
print(f"   # settings = get_settings(runtime_env='container')")

# Enable nested event loops for Jupyter notebooks (if needed for async code)
# This allows asyncio.run() to work even when an event loop is already running
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass  # Silently fail - notebooks that need it will handle the error

# Set NotePlan directory based on environment
if IS_CONTAINER:
    NOTEPLAN_DIR = Path("/noteplan")
    ENV_NAME = "🐳 Container"
else:
    NOTEPLAN_DIR = Path(os.getenv(
        "NOTEPLAN_DIR",
        "/Users/omareid/Library/Containers/co.noteplan.NotePlan3/Data/Library/Application Support/co.noteplan.NotePlan3"
    ))
    ENV_NAME = "💻 Mac (Local)"

# Initialize dependencies
dependencies = Dependencies(settings=settings)
neo4j_manager = Neo4jClientManager(settings=settings)

print(f"\n📁 NotePlan directory: {NOTEPLAN_DIR}")
print(f"   Directory exists: {NOTEPLAN_DIR.exists()}")

# Try to connect to Neo4j with better error handling
try:
    driver = neo4j_manager.get_driver()
    print(f"\n✅ Successfully connected to Neo4j")
except Exception as e:
    print(f"\n❌ Failed to connect to Neo4j: {e}")
    print(f"\n💡 Troubleshooting tips:")
    print(f"   1. Verify Neo4j Desktop is running")
    print(f"   2. Check username (default: neo4j) - current: {settings.neo4j_username}")
    print(f"   3. Check password (default: admin123) - verify it matches your Neo4j setup")
    print(f"   4. Try connecting manually: cypher-shell -a {settings.neo4j_uri} -u {settings.neo4j_username}")
    print(f"   5. Override if needed: settings = get_settings(neo4j_password='your_password', neo4j_username='your_username')")
    raise


🔍 Runtime detection: local
✅ Settings loaded:
   Neo4j URI: bolt://localhost:7687
   Neo4j Database: knowledge
   Neo4j Username: neo4j
   Neo4j Password: ********
   LiteLLM Proxy Host: localhost

💡 To override settings, see Settings class docstring:
   help(Settings)  # or help(get_settings)
   # Quick examples:
   # settings = get_settings(neo4j_password='your_password')
   # settings = get_settings(runtime_env='container')

📁 NotePlan directory: /Users/omareid/Library/Containers/co.noteplan.NotePlan3/Data/Library/Application Support/co.noteplan.NotePlan3
   Directory exists: True

✅ Successfully connected to Neo4j


## Data Manipulation Libraries


In [4]:
# Data manipulation
import pandas as pd
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_colwidth', 500)

print("✅ Data manipulation libraries imported")


✅ Data manipulation libraries imported


## Package Information

When the repository is installed in editable mode (`pip install -e .`), the package is registered as:
- **Pip package name**: `omars-knowledge-agents` (shown in `pip list`)
- **Python import name**: `knowledge_agents` (what you use in `import` statements)

The package structure is:
- `knowledge_agents` - Main package (from `src/knowledge_agents/`)
- `notes` - NotePlan utilities (from `src/notes/`)

## Optional: Neo4j Graph and LangChain (Import as Needed)

These are imported conditionally in specific notebooks that need them.
